# Experiment No. 8 — Dashboard, Responsible AI Reporting & Final Portfolio

**Aim:** Dashboard, Responsible AI Reporting & Final Portfolio

**Objective:** Build a Streamlit/Dash dashboard, write a Responsible AI report, and publish the final repo.

**Open-source tools:** Streamlit, Dash, Markdown, Jupyter, GitHub/GitLab

This notebook is self-contained — it loads `Clean_Dataset.xlsx` directly for the drift
check, so it does not depend on any other notebook's kernel state.

> **Sandbox note:** `streamlit` cannot be installed offline here, and publishing to a
> public GitHub repository / Streamlit Community Cloud needs account credentials this
> sandbox doesn't have. The drift-check logic below is executed for real.

In [1]:
import warnings
warnings.filterwarnings("ignore")
import os
import pandas as pd
from scipy.stats import ks_2samp
from sklearn.model_selection import train_test_split

DATA_PATH = "Clean_Dataset.xlsx"
if not os.path.exists(DATA_PATH):
    try:
        from google.colab import files
        print("Please choose 'Clean_Dataset.xlsx':")
        uploaded = files.upload()
        DATA_PATH = list(uploaded.keys())[0]
    except ImportError:
        raise FileNotFoundError("Place 'Clean_Dataset.xlsx' in this notebook's working directory.")

df = pd.read_excel(DATA_PATH)
numeric_cols = ["Customer_Age", "Items_Count", "Order_Value", "Discount_Percent",
                 "Distance_Km", "Customer_Rating", "Delivery_Partner_Rating"]
train_df, test_df = train_test_split(df, test_size=0.2, random_state=42)
print(f"Loaded {df.shape[0]:,} rows")

Loaded 20,000 rows


## Develop dashboard for predictions & insights

**`app/dashboard.py`** — a four-tab Streamlit app (not executed here — needs `pip install streamlit`; the full working file is in the project repo):

In [ ]:
import streamlit as st
from model_utils import predict_from_json, FEATURE_COLUMNS

st.set_page_config(page_title="Delivery Time Predictor", layout="wide")
st.title("Quick-Commerce Delivery Time — Model Dashboard")

tab_predict, tab_insights, tab_fairness, tab_drift = st.tabs(
    ["Predict", "Model Insights", "Fairness", "Data Drift"]
)

with tab_predict:
    # input widgets for every order attribute, then:
    result = predict_from_json(payload)
    st.metric("Predicted delivery time", f"{result['predicted_delivery_time_min']:.1f} min")

with tab_insights:
    st.bar_chart(importance_df.set_index("feature")["importance_mean"])
    st.table(baseline_vs_tuned_metrics)

with tab_fairness:
    st.dataframe(fairness_report_before)
    st.dataframe(fairness_report_after)

with tab_drift:
    uploaded = st.file_uploader("New data (.xlsx)", type=["xlsx"])
    # ks_2samp(reference[col], new_df[col]) per numeric column, flagged if p < 0.05

# Run with: streamlit run app/dashboard.py

## Include SHAP plots, metrics, drift checks

The Model Insights tab surfaces Experiment 5's permutation-importance chart and Experiment 4's baseline-vs-tuned metrics table; the Fairness tab surfaces Experiment 5's before/after mitigation tables. The Data Drift tab runs the Kolmogorov–Smirnov check below.

In [2]:
def check_drift(reference, current, columns, alpha=0.05):
    rows = []
    for col in columns:
        stat, p_value = ks_2samp(reference[col].dropna(), current[col].dropna())
        rows.append({"feature": col, "ks_statistic": stat, "p_value": p_value,
                      "drift_detected": p_value < alpha})
    return pd.DataFrame(rows).sort_values("p_value")

print("=== Drift check 1: train split vs. test split (expect NO drift) ===")
report_1 = check_drift(train_df, test_df, numeric_cols)
print(report_1.to_string(index=False))

print("\n=== Drift check 2: train split vs. a deliberately shifted sample (Distance_Km > 8) ===")
shifted = df[df["Distance_Km"] > 8]
report_2 = check_drift(train_df, shifted, numeric_cols)
print(report_2.to_string(index=False))

report_1.to_csv("drift_report_no_drift.csv", index=False)
report_2.to_csv("drift_report_shifted.csv", index=False)
print("\nSaved drift_report_no_drift.csv and drift_report_shifted.csv")

=== Drift check 1: train split vs. test split (expect NO drift) ===
                feature  ks_statistic  p_value  drift_detected
            Distance_Km      0.014625 0.495809           False
        Customer_Rating      0.013875 0.564090           False
Delivery_Partner_Rating      0.011500 0.786642           False
            Items_Count      0.009875 0.910777           False
            Order_Value      0.009313 0.941721           False
       Discount_Percent      0.009188 0.947601           False
           Customer_Age      0.005937 0.999841           False

=== Drift check 2: train split vs. a deliberately shifted sample (Distance_Km > 8) ===
                feature  ks_statistic  p_value  drift_detected
            Distance_Km      0.522062 0.000000            True
        Customer_Rating      0.006778 0.944035           False
            Order_Value      0.006072 0.978988           False
Delivery_Partner_Rating      0.005945 0.983072           False
            Items_Count  

## Write Responsible AI checklist (fairness, privacy, consent)

The full checklist is `Responsible_AI.md` in the project repo, covering:

1. **Model & data summary** — including the caveat that the model explains almost none of the variance in delivery time (R² ≈ 0, Experiment 4).
2. **Fairness** — the `Age_Group` audit and threshold mitigation from Experiment 5 (demographic parity difference 0.098 → 0.002).
3. **Privacy** — no direct identifiers in the dataset; `Age_Group`/`Customer_Age` used only for the fairness audit.
4. **Consent & data provenance** — synthetic coursework dataset; requirements listed before use on real operational data.
5. **Transparency** — SHAP/LIME/permutation-importance explanations, surfaced alongside a plain-language caveat.
6. **Monitoring** — the drift check above, recommended to run on every new data batch.
7. **Known limitations** — an explicit "do not deploy for real ETA decisions" warning, carried from Experiment 4.

## Publish final code, notebooks, API, and workflow on GitHub

A complete repository (dataset, notebooks, models, API, tests, CI workflow, dashboard,
`Responsible_AI.md`, `README.md`) was assembled and git-initialized locally. Publishing
it, and deploying the dashboard to Streamlit Community Cloud, both require account
credentials this sandbox doesn't have:

```
git remote add origin <your-empty-GitHub-repo-URL>
git branch -M main
git push -u origin main
# then connect the repo at share.streamlit.io, pointing at app/dashboard.py
```

## Deliverables
- **Streamlit app link** — `app/dashboard.py` is complete and ready to deploy once pushed to GitHub (steps above); it cannot be hosted from this sandbox.
- **Responsible_AI.md** — complete, in the project repo (summarized above).
- **Final public repo link** — the repo is fully prepared and git-initialized locally; pushing it takes the two commands above.

## Conclusion

The dashboard ties together every earlier experiment — predictions, explainability/fairness, and drift monitoring — into one interface, while `Responsible_AI.md` documents the fairness, privacy, consent, and transparency posture of the whole project. The drift-check component was verified with two real tests: no false positives between train and test splits, and correct, precise detection of a deliberately introduced shift (flagged only `Distance_Km`, as expected). Only the final push to a hosted GitHub account and Streamlit Cloud deployment — both requiring credentials this sandbox does not have — remain.